# Bug Bounty Hunter Agent using AutoGen

This agent uses Microsoft AutoGen framework to create a collaborative team of agents that perform automated security testing on open-source projects. The team consists of:

- **Scanner Agent**: Performs automated vulnerability scanning
- **Analyst Agent**: Analyzes findings and prioritizes vulnerabilities
- **Reporter Agent**: Creates detailed security reports
- **Fixer Agent**: Suggests or implements fixes for found vulnerabilities

Unlike CrewAI's role-based approach, AutoGen uses conversational multi-agent collaboration with code execution capabilities.

In [ ]:
# Install required packages
%pip install --upgrade --quiet pyautogen
%pip install --upgrade --quiet bandit
%pip install --upgrade --quiet safety
%pip install --upgrade --quiet semgrep
%pip install --upgrade --quiet gitpython
%pip install --upgrade --quiet requests
%pip install --upgrade --quiet aiohttp

In [ ]:
import autogen
import os
import tempfile
import subprocess
import json
import git
from pathlib import Path
import asyncio
from typing import Dict, List, Optional
from datetime import datetime

# Import helper for environment variables
import sys
sys.path.append('..')
from helpers import set_environment_variables

In [ ]:
# Set up environment variables
set_environment_variables()

# Configure AutoGen
config_list = [
    {
        "model": "claude-3-5-sonnet-20241022",
        "api_key": os.getenv("ANTHROPIC_API_KEY"),
        "api_type": "anthropic",
        "api_version": "2023-06-01"
    }
]

llm_config = {
    "config_list": config_list,
    "temperature": 0.1,
    "timeout": 120,
}

## Security Scanner Tools

Let's create tools for various security scanning operations:

In [ ]:
class SecurityScanner:
    def __init__(self, repo_path: str):
        self.repo_path = Path(repo_path)
        self.results = {}
    
    def clone_repo(self, repo_url: str, target_dir: str) -> str:
        """Clone a repository for analysis"""
        try:
            repo = git.Repo.clone_from(repo_url, target_dir)
            return f"Successfully cloned {repo_url} to {target_dir}"
        except Exception as e:
            return f"Error cloning repository: {str(e)}"
    
    def run_bandit_scan(self) -> Dict:
        """Run Bandit security scanner for Python code"""
        try:
            result = subprocess.run(
                ["bandit", "-r", str(self.repo_path), "-f", "json", "-o", "/tmp/bandit_results.json"],
                capture_output=True,
                text=True
            )
            
            if os.path.exists("/tmp/bandit_results.json"):
                with open("/tmp/bandit_results.json", "r") as f:
                    return json.load(f)
            return {"error": "No results file generated"}
        except Exception as e:
            return {"error": str(e)}
    
    def run_safety_check(self) -> Dict:
        """Check for known security vulnerabilities in dependencies"""
        try:
            requirements_files = list(self.repo_path.rglob("requirements*.txt"))
            requirements_files.extend(list(self.repo_path.rglob("Pipfile")))
            
            all_results = []
            for req_file in requirements_files:
                result = subprocess.run(
                    ["safety", "check", "-r", str(req_file), "--json"],
                    capture_output=True,
                    text=True
                )
                if result.stdout:
                    try:
                        all_results.append(json.loads(result.stdout))
                    except json.JSONDecodeError:
                        all_results.append({"raw_output": result.stdout})
            
            return {"results": all_results}
        except Exception as e:
            return {"error": str(e)}
    
    def analyze_dockerfile(self) -> Dict:
        """Analyze Dockerfile for security issues"""
        dockerfiles = list(self.repo_path.rglob("Dockerfile*"))
        issues = []
        
        for dockerfile in dockerfiles:
            try:
                with open(dockerfile, 'r') as f:
                    content = f.read()
                
                # Basic security checks
                file_issues = []
                
                if "USER root" in content:
                    file_issues.append("Running as root user")
                
                if "--privileged" in content:
                    file_issues.append("Privileged mode detected")
                
                if "curl" in content and "https://" not in content:
                    file_issues.append("Insecure HTTP download detected")
                
                if file_issues:
                    issues.append({
                        "file": str(dockerfile),
                        "issues": file_issues
                    })
            except Exception as e:
                issues.append({
                    "file": str(dockerfile),
                    "error": str(e)
                })
        
        return {"dockerfile_issues": issues}
    
    def check_secrets(self) -> Dict:
        """Simple secrets detection"""
        secret_patterns = [
            r"password\s*=\s*['\"][^'\"]+['\"]",
            r"api_key\s*=\s*['\"][^'\"]+['\"]",
            r"secret\s*=\s*['\"][^'\"]+['\"]"  
        ]
        
        import re
        findings = []
        
        for file_path in self.repo_path.rglob("*"):
            if file_path.is_file() and file_path.suffix in ['.py', '.js', '.env', '.yml', '.yaml']:
                try:
                    with open(file_path, 'r', encoding='utf-8') as f:
                        content = f.read()
                    
                    for pattern in secret_patterns:
                        matches = re.findall(pattern, content, re.IGNORECASE)
                        if matches:
                            findings.append({
                                "file": str(file_path),
                                "pattern": pattern,
                                "matches": len(matches)
                            })
                except Exception:
                    continue
        
        return {"potential_secrets": findings}

# Initialize scanner
scanner = SecurityScanner("/tmp/target_repo")

## AutoGen Agent Setup

Now let's create the AutoGen agents with specific roles and capabilities:

In [ ]:
# Scanner Agent - Performs the actual security scanning
scanner_agent = autogen.AssistantAgent(
    name="SecurityScanner",
    system_message="""You are a security scanner specialist. Your role is to:
    1. Clone and analyze repositories for security vulnerabilities
    2. Run various security scanning tools (Bandit, Safety, custom scanners)
    3. Collect raw vulnerability data
    4. Provide technical details about discovered issues
    
    You focus on the technical execution of security scans and data collection.
    Always provide detailed technical information about vulnerabilities found.""",
    llm_config=llm_config,
    code_execution_config={
        "work_dir": "/tmp/autogen_security",
        "use_docker": False,
    }
)

# Analyst Agent - Analyzes and prioritizes findings
analyst_agent = autogen.AssistantAgent(
    name="SecurityAnalyst",
    system_message="""You are a security analyst specialist. Your role is to:
    1. Analyze raw vulnerability data from security scans
    2. Prioritize vulnerabilities based on severity, exploitability, and impact
    3. Classify vulnerabilities according to CWE/CVE standards
    4. Provide business impact assessment
    5. Recommend remediation priorities
    
    You transform raw technical findings into actionable security intelligence.
    Focus on risk assessment and strategic recommendations.""",
    llm_config=llm_config,
)

# Reporter Agent - Creates detailed security reports
reporter_agent = autogen.AssistantAgent(
    name="SecurityReporter",
    system_message="""You are a security reporter specialist. Your role is to:
    1. Create comprehensive security reports from analyzed findings
    2. Write clear, actionable vulnerability descriptions
    3. Generate executive summaries for stakeholders
    4. Provide detailed technical write-ups for developers
    5. Format reports in markdown and JSON formats
    
    You translate technical security analysis into clear, professional reports
    that can be understood by both technical and non-technical stakeholders.""",
    llm_config=llm_config,
)

# Fixer Agent - Suggests fixes and improvements
fixer_agent = autogen.AssistantAgent(
    name="SecurityFixer",
    system_message="""You are a security remediation specialist. Your role is to:
    1. Analyze security vulnerabilities and their root causes
    2. Provide specific, actionable remediation steps
    3. Suggest code fixes and configuration changes
    4. Recommend security best practices for prevention
    5. Create patches or configuration examples when possible
    
    You focus on practical solutions and prevention strategies.
    Always provide concrete, implementable fixes.""",
    llm_config=llm_config,
)

# User proxy to coordinate the agents
user_proxy = autogen.UserProxyAgent(
    name="SecurityOrchestrator",
    system_message="""You coordinate the security assessment workflow.
    You delegate tasks to appropriate agents and ensure comprehensive coverage.""",
    human_input_mode="NEVER",
    code_execution_config={
        "work_dir": "/tmp/autogen_security",
        "use_docker": False,
    }
)

## Security Assessment Workflow

Let's create a function to orchestrate the security assessment:

In [ ]:
def run_security_assessment(repo_url: str, target_name: str = "target_repo"):
    """Run a comprehensive security assessment on a repository"""
    
    # Create working directory
    work_dir = Path("/tmp/autogen_security")
    work_dir.mkdir(exist_ok=True)
    
    target_dir = work_dir / target_name
    
    # Initial message to start the security assessment
    initial_message = f"""
    Please perform a comprehensive security assessment on the repository: {repo_url}
    
    Target directory: {target_dir}
    
    The workflow should be:
    1. SecurityScanner: Clone the repository and run security scans
    2. SecurityAnalyst: Analyze findings and prioritize vulnerabilities
    3. SecurityReporter: Create detailed reports
    4. SecurityFixer: Suggest remediation steps
    
    Please coordinate this assessment and provide a final summary.
    """
    
    # Create a group chat for the agents to collaborate
    groupchat = autogen.GroupChat(
        agents=[user_proxy, scanner_agent, analyst_agent, reporter_agent, fixer_agent],
        messages=[],
        max_round=20,
        speaker_selection_method="round_robin"
    )
    
    manager = autogen.GroupChatManager(
        groupchat=groupchat,
        llm_config=llm_config
    )
    
    # Start the conversation
    result = user_proxy.initiate_chat(
        manager,
        message=initial_message
    )
    
    return result

## Demo: Security Assessment

Let's demonstrate the bug bounty hunter on a sample repository:

In [ ]:
# Example: Assess a popular Python repository
demo_repo = "https://github.com/python/cpython"

print("Starting security assessment...")
print(f"Target repository: {demo_repo}")
print("\nThis will run a comprehensive security assessment using multiple agents.")
print("The agents will collaborate to scan, analyze, report, and suggest fixes.")

# Uncomment to run the actual assessment
# result = run_security_assessment(demo_repo)
# print("\nSecurity assessment completed!")

## Manual Security Scan Example

Let's demonstrate the security scanning tools directly:

In [ ]:
# Create a sample vulnerable Python file for demonstration
demo_dir = Path("/tmp/demo_vulnerable_code")
demo_dir.mkdir(exist_ok=True)

# Create a vulnerable Python file
vulnerable_code = '''
import os
import subprocess
import pickle
import hashlib

# Vulnerability 1: Hardcoded credentials
API_KEY = "sk-1234567890abcdef"
PASSWORD = "admin123"

# Vulnerability 2: SQL injection
def get_user(user_id):
    query = f"SELECT * FROM users WHERE id = {user_id}"
    return execute_query(query)

# Vulnerability 3: Command injection
def ping_host(hostname):
    cmd = f"ping -c 1 {hostname}"
    return os.system(cmd)

# Vulnerability 4: Insecure deserialization
def load_data(data):
    return pickle.loads(data)

# Vulnerability 5: Weak cryptography
def hash_password(password):
    return hashlib.md5(password.encode()).hexdigest()

# Vulnerability 6: Path traversal
def read_file(filename):
    with open(f"/uploads/{filename}", "r") as f:
        return f.read()
'''

with open(demo_dir / "vulnerable_app.py", "w") as f:
    f.write(vulnerable_code)

# Create a requirements.txt with vulnerable dependencies
vulnerable_deps = '''
django==2.0.0
requests==2.18.0
flask==0.12.0
pyyaml==3.12
'''

with open(demo_dir / "requirements.txt", "w") as f:
    f.write(vulnerable_deps)

print("Created demo vulnerable code for testing")

In [ ]:
# Run security scans on the demo code
demo_scanner = SecurityScanner(demo_dir)

print("=== Running Security Scans ===")
print("\n1. Checking for potential secrets...")
secrets_result = demo_scanner.check_secrets()
print(json.dumps(secrets_result, indent=2))

print("\n2. Running Bandit scan...")
bandit_result = demo_scanner.run_bandit_scan()
if "results" in bandit_result:
    print(f"Found {len(bandit_result['results'])} potential security issues")
    for issue in bandit_result['results'][:3]:  # Show first 3 issues
        print(f"  - {issue.get('test_name', 'Unknown')}: {issue.get('issue_text', 'No description')}")
else:
    print("Bandit scan results:", bandit_result)

print("\n3. Running Safety check...")
safety_result = demo_scanner.run_safety_check()
print(json.dumps(safety_result, indent=2))

## Features of this AutoGen Implementation

### Key Differences from CrewAI:

1. **Conversational Multi-Agent**: AutoGen uses conversation-based coordination rather than predefined workflows
2. **Code Execution**: Built-in code execution capabilities for running security tools
3. **Dynamic Interaction**: Agents can interrupt and redirect conversations based on findings
4. **Round-Robin Coordination**: Different speaker selection methods for agent coordination

### Security Assessment Capabilities:

- **Automated Vulnerability Scanning**: Bandit, Safety, custom scanners
- **Multi-Language Support**: Python, JavaScript, Docker, YAML analysis
- **Secrets Detection**: Pattern-based secret scanning
- **Collaborative Analysis**: Multiple agents work together on complex findings
- **Report Generation**: Automated security report creation

### Real-World Applications:

1. **Open Source Security**: Automated scanning of popular repositories
2. **CI/CD Integration**: Security gates in deployment pipelines
3. **Bug Bounty Assistance**: Automated first-pass vulnerability discovery
4. **Security Auditing**: Comprehensive security assessments

This implementation demonstrates AutoGen's strength in creating collaborative, code-executing agent systems that can perform complex, multi-step security assessments.